# Denoising Diffusion Probabilistic Model (DDPM) on Fashion MNIST
Implementing the Ho et al. (2020) DDPM backbone in PyTorch with a U-Net noise predictor, forward diffusion, reverse denoising, and quantitative evaluation (SSIM + FID).

In [ ]:
!nvidia-smi
%pip install torchmetrics[image] -q

In [ ]:
from __future__ import annotations
from typing import Callable, Iterable, Optional

import math
import random
import time

import matplotlib.pyplot as plt
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch import Tensor
from torch.optim import Optimizer

from torch.utils.data import DataLoader
from torchvision import datasets, transforms, utils

from torchmetrics.image import StructuralSimilarityIndexMeasure
from torchmetrics.image.fid import FrechetInceptionDistance

torch.manual_seed(42)
np.random.seed(42)

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
class HNAdam(Optimizer):
    def __init__(
        self,
        params: Iterable[Tensor],
        lr: float = 1e-3,
        betas: tuple[float, float] = (0.9, 0.99),
        eps: float = 1e-8,
        lambda_t0: Optional[float] = None,
    ) -> None:
        if params is None:
            raise ValueError("params cannot be None.")
        if lr <= 0.0:
            raise ValueError(f"Invalid learning rate: {lr}")
        if eps < 0.0:
            raise ValueError(f"Invalid epsilon value: {eps}")
        if len(betas) != 2:
            raise ValueError("betas must be a tuple of two floats")

        beta1, beta2 = betas
        if not 0.0 <= beta1 < 1.0:
            raise ValueError(f"Invalid beta1 value: {beta1}")
        if not 0.0 <= beta2 < 1.0:
            raise ValueError(f"Invalid beta2 value: {beta2}")

        if lambda_t0 is None:
            lambda_t0 = random.uniform(2.0, 4.0)
        if not 2.0 <= lambda_t0 <= 4.0:
            raise ValueError(f"lambda_t0 must be in [2, 4], got {lambda_t0}")

        defaults = {
            "lr": lr,
            "betas": (beta1, beta2),
            "eps": eps,
            "lambda_t0": lambda_t0,
            "amsgrad": False,
        }
        super().__init__(params, defaults)

        if len(self.param_groups) == 0:
            raise ValueError("optimizer got an empty parameter list")

    @torch.no_grad()
    def step(self, closure: Optional[Callable[[], Tensor]] = None) -> Optional[Tensor]:
        loss: Optional[Tensor] = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr: float = group["lr"]
            beta1, beta2 = group["betas"]
            eps: float = group["eps"]
            lambda_t0: float = group["lambda_t0"]

            for param in group["params"]:
                if param.grad is None:
                    continue

                grad = param.grad
                if grad.is_sparse:
                    raise RuntimeError("HNAdam does not support sparse gradients")

                state = self.state[param]

                if len(state) == 0:
                    state["m"] = torch.zeros_like(param, memory_format=torch.preserve_format)
                    state["v"] = torch.zeros_like(param, memory_format=torch.preserve_format)
                    state["vhat"] = torch.zeros_like(param, memory_format=torch.preserve_format)

                m_prev: Tensor = state["m"]
                v_prev: Tensor = state["v"]
                vhat_prev: Tensor = state["vhat"]

                g_t = grad

                m_t = beta1 * m_prev + (1.0 - beta1) * g_t

                g_abs = g_t.abs()
                m_prev_norm = torch.linalg.vector_norm(m_prev)
                g_abs_norm = torch.linalg.vector_norm(g_abs)
                m_max = torch.maximum(m_prev_norm, g_abs_norm)

                zero = torch.zeros((), dtype=param.dtype, device=param.device)
                ratio = torch.where(m_max > 0.0, m_prev_norm / m_max, zero)
                lambda_t = torch.as_tensor(lambda_t0, dtype=param.dtype, device=param.device) - ratio

                v_t = beta2 * v_prev + (1.0 - beta2) * g_abs.pow(lambda_t)

                if bool((lambda_t < 2.0).item()):
                    group["amsgrad"] = True

                    vhat_t = torch.maximum(vhat_prev, v_t.abs())
                    state["vhat"] = vhat_t

                    denom = vhat_t.pow(1.0 / lambda_t) + eps
                else:
                    group["amsgrad"] = False

                    denom = v_t.pow(1.0 / lambda_t) + eps

                param.addcdiv_(m_t, denom, value=-lr)

                state["m"] = m_t
                state["v"] = v_t

        return loss

In [ ]:
# Diffusion hyperparameters (Ho et al. 2020)
T = 1000
beta_start = 1e-4
beta_end = 2e-2

betas = torch.linspace(beta_start, beta_end, T, device=device)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)
alphas_cumprod_prev = torch.cat([torch.tensor([1.0], device=device), alphas_cumprod[:-1]])

sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)
sqrt_recip_alphas = torch.sqrt(1.0 / alphas)
posterior_variance = betas * (1.0 - alphas_cumprod_prev) / (1.0 - alphas_cumprod)


def get_index_from_list(vals, t, x_shape):
    batch_size = t.shape[0]
    out = vals.gather(-1, t)
    return out.reshape(batch_size, *((1,) * (len(x_shape) - 1)))


def forward_diffusion_sample(x0, t):
    noise = torch.randn_like(x0)
    sqrt_alphas_cumprod_t = get_index_from_list(sqrt_alphas_cumprod, t, x0.shape)
    sqrt_one_minus_alphas_cumprod_t = get_index_from_list(sqrt_one_minus_alphas_cumprod, t, x0.shape)
    x_t = sqrt_alphas_cumprod_t * x0 + sqrt_one_minus_alphas_cumprod_t * noise
    return x_t, noise

In [ ]:
class SinusoidalPositionEmbeddings(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim

    def forward(self, time):
        device = time.device
        half_dim = self.dim // 2
        emb = math.log(10000) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=device) * -emb)
        emb = time[:, None].float() * emb[None, :]
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        return emb


class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_emb_dim, groups=8):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.SiLU(),
            nn.Linear(time_emb_dim, out_ch)
        )
        self.block1 = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1),
            nn.GroupNorm(groups, out_ch),
            nn.SiLU()
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(out_ch, out_ch, 3, padding=1),
            nn.GroupNorm(groups, out_ch),
            nn.SiLU()
        )
        self.res_conv = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t):
        h = self.block1(x)
        time_emb = self.mlp(t)[:, :, None, None]
        h = h + time_emb
        h = self.block2(h)
        return h + self.res_conv(x)


class Downsample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.Conv2d(channels, channels, 4, 2, 1)

    def forward(self, x):
        return self.conv(x)


class Upsample(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.conv = nn.ConvTranspose2d(channels, channels, 4, 2, 1)

    def forward(self, x):
        return self.conv(x)


class SimpleUNet(nn.Module):
    def __init__(self, in_channels=1, base_channels=64, time_dim=256):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalPositionEmbeddings(time_dim),
            nn.Linear(time_dim, time_dim * 4),
            nn.SiLU(),
            nn.Linear(time_dim * 4, time_dim)
        )

        self.init_conv = nn.Conv2d(in_channels, base_channels, 3, padding=1)

        self.down1 = ResBlock(base_channels, base_channels * 2, time_dim)
        self.downsample1 = Downsample(base_channels * 2)

        self.down2 = ResBlock(base_channels * 2, base_channels * 4, time_dim)
        self.downsample2 = Downsample(base_channels * 4)

        self.bottleneck = ResBlock(base_channels * 4, base_channels * 4, time_dim)

        self.upsample1 = Upsample(base_channels * 4)
        self.up1 = ResBlock(base_channels * 8, base_channels * 2, time_dim)

        self.upsample2 = Upsample(base_channels * 2)
        self.up2 = ResBlock(base_channels * 4, base_channels, time_dim)

        self.out_conv = nn.Sequential(
            nn.Conv2d(base_channels, base_channels, 3, padding=1),
            nn.SiLU(),
            nn.Conv2d(base_channels, in_channels, 1)
        )

    def forward(self, x, t):
        t = self.time_mlp(t)
        x = self.init_conv(x)

        x1 = self.down1(x, t)
        x2 = self.downsample1(x1)

        x3 = self.down2(x2, t)
        x4 = self.downsample2(x3)

        x5 = self.bottleneck(x4, t)

        x = self.upsample1(x5)
        x = torch.cat([x, x3], dim=1)
        x = self.up1(x, t)

        x = self.upsample2(x)
        x = torch.cat([x, x1], dim=1)
        x = self.up2(x, t)

        return self.out_conv(x)

In [ ]:
batch_size = 128
num_epochs = 50
learning_rate = 2e-4
base_channels = 64
time_dim = 256

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.FashionMNIST(root="data", train=True, download=True, transform=transform)
test_dataset = datasets.FashionMNIST(root="data", train=False, download=True, transform=transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

len(train_dataset), len(test_dataset)

In [ ]:
def show_forward_diffusion():
    img, _ = train_dataset[0]
    img = img.unsqueeze(0).to(device)
    timesteps = [0, 50, 100, 200, 500, 999]

    images = []
    for t in timesteps:
        t_tensor = torch.tensor([t], device=device)
        x_t, _ = forward_diffusion_sample(img, t_tensor)
        images.append(x_t)

    images = torch.cat(images)
    images = (images + 1) / 2
    grid = utils.make_grid(images, nrow=len(timesteps))

    plt.figure(figsize=(12, 2))
    plt.imshow(grid.permute(1, 2, 0).cpu(), cmap="gray")
    plt.axis("off")
    plt.title("Forward diffusion at different timesteps")
    plt.show()


show_forward_diffusion()

In [ ]:
model = SimpleUNet(in_channels=1, base_channels=base_channels, time_dim=time_dim).to(device)
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=learning_rate,
    betas=(0.9, 0.999),
    eps=1e-8,
)
criterion = nn.MSELoss()

loss_history = []
start_time = time.time()

model.train()
for epoch in range(num_epochs):
    epoch_losses = []
    for x, _ in train_loader:
        x = x.to(device)
        t = torch.randint(0, T, (x.size(0),), device=device).long()
        x_t, noise = forward_diffusion_sample(x, t)
        noise_pred = model(x_t, t)
        loss = criterion(noise, noise_pred)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_losses.append(loss.item())

    avg_loss = float(np.mean(epoch_losses))
    loss_history.append(avg_loss)
    print(f"Epoch {epoch + 1}/{num_epochs} - loss: {avg_loss:.6f}")

training_time = time.time() - start_time
training_time

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(loss_history, marker="o")
plt.title("DDPM noise prediction loss")
plt.xlabel("Epoch")
plt.ylabel("MSE loss")
plt.grid(True)
plt.show()

In [ ]:
@torch.no_grad()
def sample_timestep(x, t):
    betas_t = get_index_from_list(betas, t, x.shape)
    sqrt_one_minus_alphas_cumprod_t = get_index_from_list(sqrt_one_minus_alphas_cumprod, t, x.shape)
    sqrt_recip_alphas_t = get_index_from_list(sqrt_recip_alphas, t, x.shape)

    model_mean = sqrt_recip_alphas_t * (x - betas_t * model(x, t) / sqrt_one_minus_alphas_cumprod_t)
    if t[0].item() == 0:
        return model_mean

    posterior_var_t = get_index_from_list(posterior_variance, t, x.shape)
    noise = torch.randn_like(x)
    return model_mean + torch.sqrt(posterior_var_t) * noise


@torch.no_grad()
def sample(model, num_samples, img_size=28):
    model.eval()
    x = torch.randn(num_samples, 1, img_size, img_size, device=device)
    for i in reversed(range(T)):
        t = torch.full((num_samples,), i, device=device, dtype=torch.long)
        x = sample_timestep(x, t)
    return x


samples = sample(model, num_samples=16)
samples = (samples + 1) / 2
samples = samples.clamp(0, 1)

grid = utils.make_grid(samples, nrow=4)
plt.figure(figsize=(4, 4))
plt.imshow(grid.permute(1, 2, 0).cpu(), cmap="gray")
plt.axis("off")
plt.title("Generated samples")
plt.show()

In [ ]:
def collect_real_samples(loader, num_samples):
    images = []
    count = 0
    for x, _ in loader:
        images.append(x)
        count += x.size(0)
        if count >= num_samples:
            break
    return torch.cat(images, dim=0)[:num_samples]


def generate_fake_samples(model, num_samples, batch_size):
    images = []
    count = 0
    while count < num_samples:
        current_bs = min(batch_size, num_samples - count)
        fake = sample(model, current_bs)
        images.append(fake.cpu())
        count += current_bs
    return torch.cat(images, dim=0)


def prepare_for_metrics(x):
    x = (x + 1) / 2
    return x.clamp(0, 1)


def to_inception_input(x):
    x = x.repeat(1, 3, 1, 1)
    x = F.interpolate(x, size=(299, 299), mode="bilinear", align_corners=False)
    return x


def update_fid_in_batches(metric, images, real, batch_size=64):
    for i in range(0, images.size(0), batch_size):
        batch = images[i : i + batch_size]
        metric.update(to_inception_input(batch), real=real)


eval_samples = 512
eval_batch_size = 64

real_images = collect_real_samples(test_loader, eval_samples).to(device)
fake_images = generate_fake_samples(model, eval_samples, eval_batch_size).to(device)

real_01 = prepare_for_metrics(real_images)
fake_01 = prepare_for_metrics(fake_images)

ssim_metric = StructuralSimilarityIndexMeasure(data_range=1.0).to(device)
ssim_value = float(ssim_metric(fake_01, real_01).item())

fid_metric = FrechetInceptionDistance(feature=2048, normalize=True).to(device)
update_fid_in_batches(fid_metric, real_01, real=True, batch_size=eval_batch_size)
update_fid_in_batches(fid_metric, fake_01, real=False, batch_size=eval_batch_size)
fid_value = float(fid_metric.compute().item())

ssim_value, fid_value

In [ ]:
from IPython.display import Markdown, display

best_loss = float(np.min(loss_history))

metrics_table = {
    "Best MSE Loss": best_loss,
    "Training Time (s)": training_time,
    "SSIM": ssim_value,
    "FID": fid_value
}

rows = [
    ("Best MSE Loss", f"{metrics_table['Best MSE Loss']:.6f}"),
    ("Training Time (s)", f"{metrics_table['Training Time (s)']:.2f}"),
    ("SSIM", f"{metrics_table['SSIM']:.6f}"),
    ("FID", f"{metrics_table['FID']:.6f}")
]

md = "| Metric | Value |\n| --- | --- |\n"
for name, value in rows:
    md += f"| {name} | {value} |\n"

display(Markdown("### Table 1. Training and evaluation summary\n" + md))